# What is a feature?

_Feature Engineering_

**A feature is a number that stands for one aspect of your raw data; features sit between the data and the model.**

Data are observations of real-world phenomena. A photo, a sentence, a purchase, a sensor reading — each is a trace of something that happened in the world.

       A model cannot work with the world directly. It works with numbers. So we need a bridge.

---

This notebook is a practice scaffold for the **What is a feature?** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

In [ ]:
# What a feature is: features sit BETWEEN raw data and the model.
# The model only sees the numbers you hand it. If those numbers don't expose
# the structure, the model fails -- even when the structure is RIGHT THERE in
# the data. The same data, with a better feature, becomes easy. That choice of
# "which numbers to hand the model" IS feature engineering.
#
# Demo: two concentric rings. Inner ring = class 0, outer ring = class 1.
#   - RAW representation (x, y): a straight-line classifier (LogisticRegression)
#     can't split them -> accuracy ~0.5 (coin flip).
#   - ONE engineered feature, the radius r = sqrt(x^2 + y^2): the same model
#     splits them almost perfectly.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# ---------------------------------------------------------------------------
# 1. Build the data (numpy, fixed rng so the numbers reproduce exactly).
#    A "ring" of radius R: pick a random angle, sit at distance ~R from origin
#    (with a little noise so it's a fuzzy band, not a perfect circle).
# ---------------------------------------------------------------------------
rng = np.random.default_rng(0)
n = 300  # points per ring

def make_ring(radius, noise):
    angle = rng.uniform(0, 2 * np.pi, n)        # random direction
    r = radius + rng.normal(0, noise, n)        # distance from center
    return np.column_stack([r * np.cos(angle), r * np.sin(angle)])

inner = make_ring(radius=1.0, noise=0.12)   # class 0
outer = make_ring(radius=2.5, noise=0.12)   # class 1

X = np.vstack([inner, outer])               # raw features: columns are x and y
y = np.array([0] * n + [1] * n)             # labels

# ---------------------------------------------------------------------------
# 3. Reproduce the PROBLEM: train the model on the RAW (x, y) features.
#    A straight line through (x, y) cannot wrap around a ring, so it's stuck
#    at chance. cross_val_score averages accuracy over 5 train/test splits.
# ---------------------------------------------------------------------------
clf = LogisticRegression()
raw = cross_val_score(clf, X, y, cv=5).mean()
print(f"RAW (x, y) accuracy:        {raw:.3f}   (chance = 0.5)")

# ---------------------------------------------------------------------------
# 4. Apply the technique: engineer ONE feature, the radius r = sqrt(x^2 + y^2).
#    Distance-from-center is exactly what separates an inner ring from an outer
#    ring -- but the model could never compute it from x and y on its own.
# ---------------------------------------------------------------------------
r = np.sqrt(X[:, 0] ** 2 + X[:, 1] ** 2).reshape(-1, 1)  # shape (N, 1)

# ---------------------------------------------------------------------------
# 5. Show the FIX: the SAME model on the engineered radius feature.
# ---------------------------------------------------------------------------
eng = cross_val_score(clf, r, y, cv=5).mean()
print(f"ENGINEERED (radius) accuracy: {eng:.3f}")

# ---------------------------------------------------------------------------
# 2 & 4 (visualize): raw rings in (x, y) | engineered radius by class.
# ---------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# RAW: a scatter of the two rings. No straight line can separate the colors.
ax1.scatter(inner[:, 0], inner[:, 1], s=12, c="tab:blue", label="class 0 (inner)")
ax1.scatter(outer[:, 0], outer[:, 1], s=12, c="tab:red", label="class 1 (outer)")
ax1.set_title(f"RAW (x, y): no line can split  (acc {raw:.2f})")
ax1.set_xlabel("x"); ax1.set_ylabel("y")
ax1.set_aspect("equal"); ax1.legend(loc="upper right", fontsize=8)

# ENGINEERED: the single radius feature, as a histogram per class.
# The two classes land in clearly separate bands -> one cut splits them.
ax2.hist(r[y == 0], bins=30, color="tab:blue", alpha=0.7, label="class 0 (inner)")
ax2.hist(r[y == 1], bins=30, color="tab:red", alpha=0.7, label="class 1 (outer)")
ax2.set_title(f"ENGINEERED radius: cleanly split  (acc {eng:.2f})")
ax2.set_xlabel("r = sqrt(x^2 + y^2)"); ax2.set_ylabel("count")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# One-line takeaway.
# ---------------------------------------------------------------------------
print(f"PROBLEM (raw): {raw:.3f}   ->   FIX (engineered): {eng:.3f}")

## Reference implementation — pandas

In [ ]:
import pandas as pd

# Online News Popularity dataset from the book's GitHub:
# github.com/alicezheng/feature-engineering-book
# (each row is one article; columns are raw measurements about it)
df = pd.read_csv('OnlineNewsPopularity.csv')
df.columns = df.columns.str.strip()          # the file has leading spaces

# --- RAW DATA: observations of a real-world phenomenon (published articles) ---
print(df[['n_tokens_content', 'shares']].head())
#    n_tokens_content  shares
# 0             219.0     593
# 1             255.0     711
# 2             211.0    1500
# 3             531.0    1200
# 4            1072.0     505

# --- A FEATURE: a numeric representation of one aspect of the raw data ---
# Feature 1: the raw word count is already numeric -> use it directly.
df['word_count'] = df['n_tokens_content']

# Feature 2: a SIMPLE engineered feature -- a binary "is this a long article?"
# (1 if the article is longer than the median, else 0). One number per article.
median_len = df['n_tokens_content'].median()
df['is_long'] = (df['n_tokens_content'] > median_len).astype(int)

print(df[['word_count', 'is_long']].head())
#    word_count  is_long
# 0       219.0        0
# 1       255.0        0
# 2       211.0        0
# 3       531.0        1
# 4      1072.0        1

# The model never sees the raw article -- only feature columns like these.
X = df[['word_count', 'is_long']]   # what the model consumes
y = df['shares']                    # the target we want to predict

## Visualize the data & results

_On real tumor data, does a simple feature transform (scaling) actually change how well the model does?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()          # 569 real tumors, 30 numeric features
X, y = data.data, data.target
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

# --- Model on the RAW features (very different column scales) ---
clf_raw = LogisticRegression(max_iter=200).fit(Xtr, ytr)
raw_acc = clf_raw.score(Xte, yte)

# --- Model on a SIMPLE engineered feature: standardized columns ---
scaler = StandardScaler().fit(Xtr)
clf_scaled = LogisticRegression(max_iter=200).fit(scaler.transform(Xtr), ytr)
scaled_acc = clf_scaled.score(scaler.transform(Xte), yte)

print(round(raw_acc, 3), round(scaled_acc, 3))
# -> 0.936 0.959   (same model + data; only the features changed)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your text classifier does poorly. A teammate says "let's swap the logistic regression for a deep neural network". According to Chapter 1's framing, why might that be the wrong first move, and what should you check instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the pipeline order: raw data &rarr; features &rarr; model &rarr; prediction. — _The model only ever sees the features, never the raw data._
- Ask whether the features carry the signal. If they do not, the model cannot recover it. — _Garbage in, garbage out — a fancier model cannot invent information the features left out._
- Inspect and improve the features first: better tokenization, tf-idf weighting, n-grams, dropping noise. — _Most projects gain more from better features than from a bigger model._

**Answer:** A more powerful model can't fix bad features. Check the features first — the model only sees what feature engineering puts in front of it. Improve the features before reaching for a bigger model.

</details>

**Problem 2.** The raw review is "good food good price". You build two features: $\phi_1$ = count of the word "good", and $\phi_2$ = total word count. What feature vector $\mathbf{z}$ does the model receive?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply $\phi_1$: count "good". It appears twice, so $\phi_1(x) = 2$. — _Each feature function returns one number extracted from the raw text._
- Apply $\phi_2$: count all words. "good food good price" has 4 words, so $\phi_2(x) = 4$. — _A second feature captures a different aspect of the same raw observation._
- Stack them into the feature vector $\mathbf{z} = [\phi_1(x), \phi_2(x)] = [2, 4]$. — _The vector $\mathbf{z}$ is exactly what the model sees — the raw text is gone._

**Answer:** $\mathbf{z} = [2, 4]$.

</details>

**Problem 3.** You add a feature scaling step that rescales every column to a similar range. It noticeably helps your logistic regression but does almost nothing for your gradient-boosted trees. Why does the same feature transform help one model and not the other?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall how a linear model uses features: it multiplies each by a weight and adds them, so the scale of a feature directly affects its influence and the optimizer. — _Wildly different scales let one column dominate and slow or distort fitting — Chapter 1's point that the right feature depends on the model._
- Recall how a tree uses features: it splits on whether a feature is above or below a threshold, caring only about order. — _A monotone rescaling does not change the order, so the same splits remain available._
- Conclude that scaling matters for the scale-sensitive model but is invisible to the order-only model. — _The best feature engineering choice depends on the model you will use._

**Answer:** Scaling helps the linear model because it weights features by their raw scale; trees split only on order, so a monotone rescale leaves them unchanged. The right feature depends on the model.

</details>